# 🌾 Module A: Crop Recommendation using Multiclass Classification
*Smart Crop & Agriculture Analyzer*


In [ ]:
# =============================================================================
# 1. ENVIRONMENT SETUP & CORE LIBRARY IMPORTS
# =============================================================================
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn Preprocessing & Pipelines
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Scikit-Learn Models
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, mean_absolute_error, mean_squared_error,
    r2_score, silhouette_score
)

# Plotting & Display configurations
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

print("✅ All core Machine Learning and Data Science libraries imported successfully!")


## 1. Load Crop Recommendation Dataset


In [ ]:
# 2.2 Inspect Crop Recommendation Data Quality
print("=== Crop Recommendation Dataset (First 5 Rows) ===")
display(crop_df.head())

print("\nMissing Values Check:")
print(crop_df.isnull().sum())

print(f"\nDuplicate Records: {crop_df.duplicated().sum()}")
print(f"Total Unique Crop Categories: {crop_df['label'].nunique()} crops")
print(f"Crop Categories: {sorted(crop_df['label'].unique().tolist())}")

print("\nClass Distribution (Balanced: 100 samples per crop):")
print(crop_df['label'].value_counts().head(5))


In [ ]:
# 2.3 Feature Descriptive Statistics
print("=== Descriptive Statistics for Soil Nutrients and Climate Features ===")
display(crop_df.describe().round(2))


In [ ]:
# 2.4 Correlation Matrix Heatmap
plt.figure(figsize=(8, 5))
numeric_cols = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
corr_matrix = crop_df[numeric_cols].corr()

sns.heatmap(corr_matrix, annot=True, cmap="Greens", fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix of Soil & Weather Parameters", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


---
## 🧹 3. Data Cleaning, Preprocessing & Preventing Data Leakage

In raw agricultural government statistics:
- **Unit Issue:** Crops like Coconut report production in count of nuts (e.g. 65 million nuts from 18,000 ha), while other crops report in metric tonnes. Mixing nuts with tonnes corrupts yield modeling; hence non-tonne crops are removed.
- **Yield Calculation:** $\text{Yield} = \frac{\text{Production (tonnes)}}{\text{Area (hectares)}}$.
- **Outlier Cleaning:** Scale reporting anomalies (e.g., entering kilograms instead of tonnes resulting in yields > 10,000 t/ha) are filtered to realistic agronomic bounds ($0 < \text{Yield} \le 100\text{ t/ha}$).
- **Data Leakage Prevention:** Preprocessing transformers (`StandardScaler`, `OneHotEncoder`) are fitted **strictly on training data** (`X_train`) and applied (`transform()`) to test data (`X_test`).


## 2. Train-Test Split (Stratified 80/20)


In [ ]:
X = crop_df[["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]]
y = crop_df["label"].str.strip().str.lower()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train samples: {len(X_train)} | Test samples: {len(X_test)}")


## 3. Train Classifiers: Random Forest, Decision Tree, KNN


In [ ]:
# 4.1 Training Classification Models
clf_models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, random_state=42),
    "K-Nearest Neighbors (KNN)": Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=5, weights="distance"))
    ])
}

clf_metrics = {}

for name, model in clf_models.items():
    model.fit(X_crop_train, y_crop_train)
    y_pred = model.predict(X_crop_test)
    
    acc = accuracy_score(y_crop_test, y_pred)
    prec = precision_score(y_crop_test, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_crop_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_crop_test, y_pred, average="weighted", zero_division=0)
    cm = confusion_matrix(y_crop_test, y_pred)
    
    clf_metrics[name] = {
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "Confusion_Matrix": cm,
        "Classes": sorted(list(set(y_crop_test)))
    }

# Display Model Comparison Table
clf_summary_df = pd.DataFrame([
    {
        "Algorithm": k,
        "Accuracy": f"{v['Accuracy']*100:.2f}%",
        "Precision": f"{v['Precision']*100:.2f}%",
        "Recall": f"{v['Recall']*100:.2f}%",
        "F1-Score": f"{v['F1-Score']*100:.2f}%"
    } for k, v in clf_metrics.items()
])
print("=== Classification Model Performance Comparison on Test Set (20%) ===")
display(clf_summary_df)


In [ ]:
# 4.2 Visualizing Classifier Performance Metrics
plt.figure(figsize=(8, 4))
plot_metrics = pd.DataFrame([
    {"Model": k, "Accuracy": v["Accuracy"], "Precision": v["Precision"], "Recall": v["Recall"], "F1-Score": v["F1-Score"]}
    for k, v in clf_metrics.items()
])
plot_df = pd.melt(plot_metrics, id_vars="Model", var_name="Metric", value_name="Score")

sns.barplot(data=plot_df, x="Model", y="Score", hue="Metric", palette="Greens")
plt.ylim(0.9, 1.01)
plt.title("Classification Benchmark Comparison on 20% Test Split", fontsize=11, fontweight="bold")
plt.ylabel("Score")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:
# 4.3 Confusion Matrix for Random Forest Classifier
best_clf_name = "Random Forest"
rf_cm = clf_metrics[best_clf_name]["Confusion_Matrix"]
rf_classes = clf_metrics[best_clf_name]["Classes"]

plt.figure(figsize=(10, 8))
sns.heatmap(rf_cm, annot=True, fmt="d", cmap="Greens", xticklabels=rf_classes, yticklabels=rf_classes, annot_kws={"size": 7})
plt.title(f"Confusion Matrix: {best_clf_name} (22 Crop Categories, 440 Test Samples)", fontsize=11, fontweight="bold")
plt.xlabel("Predicted Crop")
plt.ylabel("Actual True Crop")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()


## 4. Prediction Function with Agronomic Explanation


In [ ]:
# 4.4 Crop Recommendation Prediction Function with Agronomic Explanation
best_crop_model = clf_models[best_clf_name]

def recommend_crop(n, p, k, temperature, humidity, ph, rainfall):
    """Predicts optimal crop and explains input soil/weather conditions."""
    input_df = pd.DataFrame([[n, p, k, temperature, humidity, ph, rainfall]],
                            columns=["N", "P", "K", "temperature", "humidity", "ph", "rainfall"])
    
    predicted_crop = best_crop_model.predict(input_df)[0]
    probabilities = best_crop_model.predict_proba(input_df)[0]
    confidence = probabilities[list(best_crop_model.classes_).index(predicted_crop)] * 100
    
    # Agronomic Explanations
    n_desc = "High (Optimal for nitrogen feeders like rice, maize, cotton)" if n > 80 else ("Low (Legumes favored)" if n < 40 else "Medium")
    p_desc = "High (Ideal for phosphorus-accumulating fruits like apples, grapes)" if p > 70 else ("Low" if p < 30 else "Optimal")
    k_desc = "Very High (Supports heavy fruit formation)" if k > 60 else ("Adequate" if k >= 30 else "Low")
    ph_desc = "Acidic (Favors tea, coffee, potatoes)" if ph < 5.5 else ("Alkaline (Favors cotton, barley)" if ph > 7.5 else "Neutral (Ideal across most crops)")
    rain_desc = "High Rainfall (Ideal for wetland rice, jute)" if rainfall > 150 else ("Low Rainfall (Millets, pulses favored)" if rainfall < 60 else "Moderate Rainfall")
    
    print(f"🎯 RECOMMENDED CROP: {predicted_crop.upper()} (Confidence: {confidence:.1f}%)")
    print(f"   • Model Used: {best_clf_name} (Test Accuracy: {clf_metrics[best_clf_name]['Accuracy']*100:.2f}%)")
    print("   • Agronomic Interpretation:")
    print(f"     - Nitrogen (N: {n}): {n_desc}")
    print(f"     - Phosphorus (P: {p}): {p_desc}")
    print(f"     - Potassium (K: {k}): {k_desc}")
    print(f"     - Soil pH ({ph}): {ph_desc}")
    print(f"     - Rainfall ({rainfall} mm): {rain_desc}")
    return predicted_crop

# Test Sample 1: Wetland Rice Conditions
print("--- TEST CASE 1: High rainfall & nitrogen ---")
recommend_crop(n=90, p=42, k=43, temperature=21.0, humidity=82.0, ph=6.5, rainfall=203.0)

print("\n--- TEST CASE 2: High potassium orchard conditions ---")
recommend_crop(n=20, p=134, k=199, temperature=22.0, humidity=92.0, ph=5.9, rainfall=112.0)
